In [1]:
"""
Zebra Puzzle (Einstein's Riddle) solved as a CSP.
Implementation of Backtracking Search with MRV Heuristic and Node Consistency.
Representation: Attributes are Variables, House positions (1-5) are Domains.
"""

class ZebraCSP:
    def __init__(self):
        # 1. Categories definition for Global Constraints (Alldiff)
        self.categories = [
            ['Englishman', 'Spaniard', 'Norwegian', 'Ukrainian', 'Japanese'],
            ['Red', 'Green', 'Ivory', 'Yellow', 'Blue'],
            ['Water', 'Tea', 'Milk', 'OrangeJuice', 'Coffee'],
            ['Hershey', 'KitKats', 'Smarties', 'Snickers', 'MilkyWays'],
            ['Dog', 'Snails', 'Fox', 'Horse', 'Zebra']
        ]

        # Flatten categories to get all variables
        self.variables = [var for category in self.categories for var in category]

        # 2. Domains: Each variable can initially be in any house from 1 to 5
        self.domains = {var: [1, 2, 3, 4, 5] for var in self.variables}

        # 3. Node Consistency (Pre-processing Unary Constraints)
        # We reduce the domain of variables with known absolute positions before searching
        self.domains['Norwegian'] = [1]  # Clue 3: Norwegian in the first house
        self.domains['Milk'] = [3]       # Clue 14: Milk is drunk in the middle house

    def is_consistent(self, var, value, assignment):
        """Checks if assigning 'value' to 'var' violates any constraints."""

        # A. Global Constraint (Alldiff): Variables in the same category must have different houses
        for category in self.categories:
            if var in category:
                for other_var in category:
                    if other_var != var and other_var in assignment and assignment[other_var] == value:
                        return False

        # Create a temporary assignment to test relationships
        test_assign = assignment.copy()
        test_assign[var] = value

        # Helper functions for Binary and Spatial Constraints
        def equal(v1, v2):
            if v1 in test_assign and v2 in test_assign:
                return test_assign[v1] == test_assign[v2]
            return True

        def next_to(v1, v2):
            if v1 in test_assign and v2 in test_assign:
                return abs(test_assign[v1] - test_assign[v2]) == 1
            return True

        def right_of(v1, v2):
            if v1 in test_assign and v2 in test_assign:
                return test_assign[v1] == test_assign[v2] + 1
            return True

        # B. Binary Constraints (Equivalences)
        if not equal('Englishman', 'Red'): return False
        if not equal('Spaniard', 'Dog'): return False
        if not equal('Coffee', 'Green'): return False
        if not equal('Ukrainian', 'Tea'): return False
        if not equal('Smarties', 'Snails'): return False
        if not equal('Snickers', 'OrangeJuice'): return False
        if not equal('Japanese', 'MilkyWays'): return False
        if not equal('KitKats', 'Yellow'): return False

        # C. Spatial / Positional Constraints
        if not right_of('Green', 'Ivory'): return False
        if not next_to('Norwegian', 'Blue'): return False
        if not next_to('Hershey', 'Fox'): return False
        if not next_to('KitKats', 'Horse'): return False

        return True

    def select_unassigned_variable_mrv(self, assignment):
        """
        Heuristic: Minimum Remaining Values (MRV)
        Selects the unassigned variable with the fewest valid remaining values.
        """
        unassigned_vars = [v for v in self.variables if v not in assignment]

        best_var = None
        min_valid_values = float('inf')

        for var in unassigned_vars:
            valid_count = 0
            for val in self.domains[var]:
                if self.is_consistent(var, val, assignment):
                    valid_count += 1

            if valid_count < min_valid_values:
                min_valid_values = valid_count
                best_var = var

        return best_var

    def backtrack(self, assignment):
        """Core Recursive Backtracking Search Algorithm."""
        # Base case: All variables assigned successfully
        if len(assignment) == len(self.variables):
            return assignment

        # Select next variable using MRV heuristic instead of static ordering
        var = self.select_unassigned_variable_mrv(assignment)

        # Iterate over the domain of the selected variable
        for value in self.domains[var]:
            if self.is_consistent(var, value, assignment):
                assignment[var] = value

                # Deepen the search recursively
                result = self.backtrack(assignment)
                if result is not None:
                    return result

                # Backtrack: remove assignment if it leads to a dead end
                del assignment[var]

        return None

def print_solution(solution):
    """Formats and prints the results to directly answer the assignment questions."""
    if not solution:
        print("No solution found due to over-constrained environment.")
        return

    # Invert the dictionary to map houses to their attributes
    houses = {1: {}, 2: {}, 3: {}, 4: {}, 5: {}}
    categories_names = ['Nationality', 'Color', 'Drink', 'Candy', 'Pet']

    # Map each variable back to its category name and house number
    for category_idx, category in enumerate(csp.categories):
        for var in category:
            house_num = solution[var]
            houses[house_num][categories_names[category_idx]] = var

    print("=== FULL ZEBRA PUZZLE STATE ===")
    for h in range(1, 6):
        print(f"House {h}: {houses[h]}")

    print("\n=== ASSIGNMENT QUESTIONS ANSWERS ===")
    # Extract specific answers programmatically
    water_house = solution['Water']
    zebra_house = solution['Zebra']

    water_drinker = [var for var in csp.categories[0] if solution[var] == water_house][0]
    zebra_owner = [var for var in csp.categories[0] if solution[var] == zebra_house][0]

    print(f"1. In which house do they drink water? -> House {water_house} (The {water_drinker})")
    print(f"2. Where does the zebra live?          -> House {zebra_house} (With the {zebra_owner})")

# Execution block
if __name__ == '__main__':
    print("Initializing CSP Solver with MRV and Node Consistency...")
    csp = ZebraCSP()
    final_solution = csp.backtrack({})
    print_solution(final_solution)

Initializing CSP Solver with MRV and Node Consistency...
=== FULL ZEBRA PUZZLE STATE ===
House 1: {'Nationality': 'Norwegian', 'Color': 'Yellow', 'Drink': 'Water', 'Candy': 'KitKats', 'Pet': 'Fox'}
House 2: {'Nationality': 'Ukrainian', 'Color': 'Blue', 'Drink': 'Tea', 'Candy': 'Hershey', 'Pet': 'Horse'}
House 3: {'Nationality': 'Englishman', 'Color': 'Red', 'Drink': 'Milk', 'Candy': 'Smarties', 'Pet': 'Snails'}
House 4: {'Nationality': 'Spaniard', 'Color': 'Ivory', 'Drink': 'OrangeJuice', 'Candy': 'Snickers', 'Pet': 'Dog'}
House 5: {'Nationality': 'Japanese', 'Color': 'Green', 'Drink': 'Coffee', 'Candy': 'MilkyWays', 'Pet': 'Zebra'}

=== ASSIGNMENT QUESTIONS ANSWERS ===
1. In which house do they drink water? -> House 1 (The Norwegian)
2. Where does the zebra live?          -> House 5 (With the Japanese)
